# Reinforcement Learning 
Imagine a child learning to ride a bike. The child does not know what happens exactly when wheels interact with the ground and may have to experience many failures before learning to keep a balance. Implicitly, the child may reveal that steering in certain ways will cause a failure after a few seconds and how to make instaneous corrections to ride as far as possible. The process might be accelerated by looking at how other people ride or receiving aids from parents. But it is still largely a trial-and-error process, in which the child gradually builds from experience for cycling over various terrains. 

The computational model emulates this process is called _Reinforcement Learning_ (RL). It is an appealing method for building intelligent agents, e.g. a robot, by having desired behaviours emerged in a self-organized way, minimizing the programming efforts of human experts. This module will cover a sketch of practical RL. The note will focus on its general formulation, notations, overall methodologies, popular software libraries and interface for quickly prototyping and solving an RL problem on your own computer. The description is largely high-level while still contains some main mathematical expressions. Numerous excellent tutorials can be found online and a more formal treatment can be found in seminal textbooks such as {cite:p}`Sutton1998`.


## Reinforcement Learning Problem

RL systems are defined by a tuple of elements $\{\mathcal{X}, \mathcal{U}, \mathcal{T}, \mathcal{R}, \rho_0 \}$. Here $\mathcal{X}$ stands for the set of possible agent states i.e. $\mathbf{x} \in \mathcal{X}$, for instance the position and velocity of the child and bike in the above example. $\mathbf{u} \in \mathcal{U}$ is the action taken to steer the evolution of the state, e.g., the forces exerted on the bike pedals. Numerically $\mathbf{x}$ and $\mathbf{u}$ are usually vectors whose entries include quantities that the agent can acquire and implement. The notation uses bolded symbol to highlight this fact. 

:::{note}
We are mixing the notations here by using $\mathbf{x}$ and $\mathbf{u}$, which are more common in automatic control. We do this because the course will largely concern about progamming real robot systems. You may find $\mathbf{s}$ for states and $\mathbf{a}$ for actions in many computer science literature. 
:::

The state evolution is an instance of $\mathcal{T}$, describing how a state at the $t$ step $\mathbf{x}_t$ will transit to another state at the $t+1$ step, $\mathbf{x}_{t+1}$, when the agent takes an action $\mathbf{u}_t$. This can represent particularly complex processes such as the physics involving the child, bicycle and interaction between the tire and the ground. The enormeous complexities and stochasticity often lead to a non-deterministic process $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ so the next-step state can be only treated as a sample taken from a random distribution conditioned on $\mathbf{x}_t$ and $\mathbf{u}_t$. Nevertheless, once there is a rule to decide which action to take, often called control _policy_ and noted as $\pi(\mathbf{u}_t|\mathbf{x}_t)$, we can recursively apply this policy to temporally unroll the process $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ and retrieve a _rollout_ of states and actions $\zeta = \{\mathbf{x}_0, \mathbf{u}_0, \mathbf{x}_1, \mathbf{u}_1, ... \}$. The desirability of the agent behaviour manifested by rollouts is captured by a function $r(\mathbf{x}_t, \mathbf{u}_t) \in \mathcal{R}$, which assigns a scalar score to the actions and states at time $t$. The score is usually made higher e.g., when the child's position is not so low to be considered as falling over so a policy yielding these states will be rewarded. Thus $r(\mathbf{x}_t, \mathbf{u}_t)$ is called _reward function_ and the goal of RL is to find an _optimal policy_ that can maximize the accumulated rewards, a.k.a the expected return, from realized rollouts. Formally, this can be written as maximizing an objective

$$
    \mathcal{J}_{\pi} = \mathbb{E}_{\mathbf{x}_0 \sim \rho_0(\mathbf{x}_0), \mathbf{u}_t \sim \pi(\mathbf{u}_t|\mathbf{x}_t), \mathbf{x}_t \sim p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)} [\sum\limits_{t} r(\mathbf{x}_t, \mathbf{u}_t)]
$$
where $\rho_0(\mathbf{x}_0)$ is the last element of the tuple $\{\mathcal{X}, \mathcal{U}, \mathcal{T}, \mathcal{R}, \rho_0 \}$, denoting the possible initial states that the agent starts with. The connections between these elements can be illustrated by the following diagram.

```{figure} ../images/mdp_diagram.png
---
name: rl-diagram
---

A general diagram of reinforcement learning problem.
```

RL algorithms solve this problem by _learning_ the optimal policy. Here learning means the process relies on experience or data that in most cases are a couple of data snippets $\{\mathbf{x}_t, \mathbf{u}_t, \mathbf{x}_{t+1}, r_t \}$. Note that retrieving these snippets only requires to sample from $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ and $r(\mathbf{x}_t, \mathbf{u}_t)$. This removes the necessity of knowing the exact analytical forms of state transition, e.g. physics for interaction between wheels and ground. Most RL algorithms thus iterate between two stages:

- Using a policy (not necessarily the one being learned) to sample rollouts of $\{\mathbf{x}_t, \mathbf{u}_t, \mathbf{x}_{t+1}, r_t \}$;
- Using the rollout data to improve the learning policy;

RL problems are hard because the effects of agent action are not immediate: entering into a state with a high reward may not necessarily benefit the accumulated return in the longrun. Identifying the relation between an action to take and a return that can be hugely delayed is thus critical to finding the optimal policy, often addressed as a _credit assignment problem_. The space of possible policies can be very large. Recall the vector representations of $\mathbf{x}$ and $\mathbf{u}$. We may use deep neural networks as a policy taking $\mathbf{x}$ as input and outputing $\mathbf{u}$ so the policy can model very complicated rules. This may entail learning a large model from a huge amount of data snippets $\{\mathbf{x}_t, \mathbf{u}_t, \mathbf{x}_{t+1}, r_t \}$, resembling challenges faced in other deep learning domains.




## Solving RL - Model-free Policy Gradient

A policy can be represented as a mathematical function that maps $\mathbf{x}$ to $\mathbf{u}$, e.g. as discussed in the neural network example above. In this case, we may rewrite the objective by highlighting the function parameter as $\pi_{\theta}$, given it fully determines the policy $\pi$ and hence the agent bebhaviour:

$$
\label{eq-rlobjective}
\mathcal{J}_{{\color{red}\theta}} = \mathbb{E}_{\mathbf{x}_0 \sim \rho_0(\mathbf{x}_0), \mathbf{u}_t \sim \pi_{{\color{red}\theta}}(\mathbf{u}_t|\mathbf{x}_t), \mathbf{x}_{t+1} \sim p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)} [\sum\limits_{t} r(\mathbf{x}_t, \mathbf{u}_t)]
$$

If the function is differentiable with respect to $\theta$, one may use $\nabla_{\theta} \mathcal{J}_{\pi_{\theta}}$, called _policy gradient_, to iteratively update the target policy $\pi_{\theta}$ as in gradient-based optimization. This may appear unreasonable for the absence of analytical equations $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ and $r(\mathbf{x}_t, \mathbf{u}_t)$. How can the chain rule of derivatives apply here? In fact, under a mild assumption about the transition model, one can obtain an estimate of $\nabla_{\theta} \mathcal{J}_{\pi_{\theta}}$ to develop a _model-free_ algorithm:

$$
\label{eq-pgreinforce}
\begin{aligned}
\nabla_{\theta} \mathcal{J}_{\pi_{\theta}} = & \nabla_{\theta} \int \rho_0(\mathbf{x} _0) \prod\limits_{t=1}^{T} [\pi_{\theta}(\mathbf{u}_{t-1}|\mathbf{x}_{t-1}) p(\mathbf{x}_{t}|\mathbf{x}_{t-1}, \mathbf{u}_{t-1})] \pi_{\theta}(\mathbf{u}_{T}|\mathbf{x}_{T})  \sum\limits_{t=0}^T r(\mathbf{x}_t, \mathbf{u}_t) d \mathbf{x}_{0:T} d \mathbf{u}_{0:T}    \\
= & \int p_{\theta}(\zeta) \nabla_{\theta} \log p_{\theta}(\zeta) \sum\limits_{t=0}^T r(\mathbf{x}_t, \mathbf{u}_t) d \mathbf{x}_{0:T} d \mathbf{u}_{0:T}   \\
= & \mathbb{E}_{p_{\theta} (\zeta)} [ \nabla_{\theta} (\log \rho_0(\mathbf{x}_0)  + \sum\limits_{t=0}^T \log \pi_{\theta}(\mathbf{u}_{t}|\mathbf{x}_{t}) + \sum\limits_{t=1}^{T} \log p(\mathbf{x}_{t}|\mathbf{x}_{t-1}, \mathbf{u}_{t-1}) ) \sum\limits_{\tau=0}^T r(\mathbf{x}_{\tau}, \mathbf{u}_{\tau}) ]   \\
= & \mathbb{E}_{p_{\theta} (\zeta)} [ ( \underbrace{\nabla_{\theta} \log \rho_0(\mathbf{x}_0)}_{=0}  + \sum\limits_{t=0}^T \nabla_{\theta} \log \pi_{\theta}(\mathbf{u}_{t}|\mathbf{x}_{t}) + \sum\limits_{t=1}^{T} \underbrace{\nabla_{\theta} \log p(\mathbf{x}_{t}|\mathbf{x}_{t-1}, \mathbf{u}_{t-1})}_{=0} ) \sum\limits_{\tau=0}^T r(\mathbf{x}_{\tau}, \mathbf{u}_{\tau}) ]  \\
= & \mathbb{E}_{p_{\theta} (\zeta)} [ \sum\limits_{t=0}^T \nabla_{\theta} \log \pi_{\theta}(\mathbf{u}_{t}|\mathbf{x}_{t}) \sum\limits_{\tau=0}^T r(\mathbf{x}_{\tau}, \mathbf{u}_{\tau}) ]
\end{aligned}
$$
where the probability of rollouts $p_{\theta} (\zeta) = \rho_0(\mathbf{x} _0) \prod\limits_{t=1}^{T} [\pi_{\theta}(\mathbf{u}_{t-1}|\mathbf{x}_{t-1}) p(\mathbf{x}_{t}|\mathbf{x}_{t-1}, \mathbf{u}_{t-1})] \pi_{\theta}(\mathbf{u}_{T}|\mathbf{x}_{T})$. The second line of derivation uses the trick of logartihm to turn the product to summation with $\nabla_{\theta} p_{\theta}(\zeta) = \frac{p_{\theta}(\zeta)}{p_{\theta}(\zeta)} \nabla_{\theta} p_{\theta}(\zeta) = p_{\theta}(\zeta) \nabla_{\theta} \log p_{\theta}(\zeta)$. The extra $p_{\theta}(\zeta)$ will allow to write the evaluation of gradient as expectation so we can obtain an estimate of $\nabla_{\theta} \mathcal{J}_{\pi_{\theta}}$ by simply taking rollout samples. The other term $\nabla_{\theta} \log\pi_{\theta}$ needs to know $\pi_{\theta}$ which is up to our design. 

Intuitively, the last line looks at the log-likelihood of decisions made in rollouts with realized return $\sum\limits_{\tau=0}^T r(\mathbf{x}_{\tau}, \mathbf{u}_{\tau})$ as sample weights. Those yield higher return will thus become more likely to be sampled after $\nabla_{\theta} \mathcal{J}_{\pi_{\theta}}$ is applied. This is called _REINFORCE_{cite:p}`Williams_Reinforce92` and underpins many modern policy-gradient-based reinforcement learning algorithms, such as Proximal Policy Optimization (PPO){cite:p}`schulman2017ppo`. 

The fact of having an estimate of $\nabla_{\theta} \mathcal{J}_{\pi_{\theta}}$ begs the question that how bad this estimate can be. It is known that REINFORCE is _unbiased_ which means the exact $\nabla_{\theta} \mathcal{J}_{\pi_{\theta}}$ can be recovered given sufficient samples. However, it can be a very large number of samples and remember that is just for one step of gradient update. Many developments thus attempt to mitigate this by reducing the _variance_ of the estimate, sometimes having to accept the loss of unbiasness.

:::{note}
Policy-gradient is not a gradient in the sense of taking the first-order derivative of the optimization objective. Indeed, one can see that the estimate [](#eq-pgreinforce) is purely based on function evaluation of $r$ and resembles similarities to other zero-order algorithms such as evolution strategies. Some recent findings argue it might not be as problematic as it appears: the cost surface induced by RL rewards can be particularly jagged due to hard constraints and repetitive nonlinear transformations. Random sampling could smooth the landscape {cite:p}`bettergrad_icml22`. It could also be more favoured than following exact gradients for more exploration of the state space. See [relevant discussion](https://arxiv.org/abs/2403.14864).
:::

## Solving RL - Model-free Value-based Methods 

<!-- Note: this might not be necessary given learning objectivies do not include this. For now, put it here for completeness of the topic.

Optimal sub-structure in MDP problems; Bellman equations and dynamic programming, notations for value functions, temporal-difference errors. Showing optimal policy can be determined by knowing optimal value/value-action functions.

Derive Q-learning with deep nets as function approximators.

Remarks about actor-critic architecture, convergence/contraction of fix-point iterations. Link to practical algorithms such as DQN(Rainbow) and SAC. -->

The objective [](#eq-rlobjective) can be re-written as integrating a _value function_ over the initial distribution, i.e. {math}`\mathcal{J}_{\pi_{\theta}} = \int_{\rho_0(\mathbf{x})} V^{\pi_{\theta}}(\mathbf{x})d \mathbf{x}`. Here the value function $V^{\pi_{\theta}}$ gives a compact form for evaluating the return of _starting from state $\mathbf{x}$ and following the policy $\pi_{\theta}$ for remaining steps_. This implies the value function can be defined in a recursive manner if the return is evaluated as the sum of current step reward and the return after that, with explicit time step indices:
$$
\label{eq-valuefunc}
V^{\pi_{\theta}}(\mathbf{x}_t) = \mathbb{E}_{\pi_{\theta}(\mathbf{u}_t | \mathbf{x}_t )} [ r(\mathbf{x}_t, \mathbf{u}_t) + \mathbb{E}_{p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)}[V^{\pi_{\theta}}(\mathbf{x}_{t+1})]]
$$
An immediate outcome from this observation is that, one can evaluate how good $\pi_{\theta}$ is by finding $V^{\pi_{\theta}}$ that admits the above equation. The recursion structure can be exploited to boostrap an initial estimation. For instance, if the estimate after $k$ iterations was denoted as $\hat{V}^{\pi_{\theta}}_k$, a rule of updating it to the $k+1$ iterations can be written as:

$$
\label{eq-policyeval}
\hat{V}^{\pi_{\theta}}_{k+1}(\mathbf{x}_{t}) \leftarrow \hat{V}^{\pi_{\theta}}_k(\mathbf{x}_{t}) + \alpha (r(\mathbf{x}_t, \mathbf{u}_t) + \hat{V}^{\pi_{\theta}}_k(\mathbf{x}_{t+1}) - \hat{V}^{\pi_{\theta}}_k(\mathbf{x}_{t}))
$$

The update needs to be applied to the entire state space while empirically this is done with sample snippets $\{\mathbf{x}_t, \mathbf{u}_t, \mathbf{x}_{t+1}, r_t \}$. $\alpha$ controls the strength of overwritting the old estimate with bootstrapped $r(\mathbf{x}_t, \mathbf{u}_t) + \hat{V}^{\pi_{\theta}}_k(\mathbf{x}_{t+1})$. When $\alpha = 1$, one can recover an update rule similar to [dynamic programming](https://en.wikipedia.org/wiki/Dynamic_programming) algorithm and expect the iterations to converge as the residual $r(\mathbf{x}_t, \mathbf{u}_t) + \hat{V}^{\pi_{\theta}}_k(\mathbf{x}_{t+1}) - \hat{V}^{\pi_{\theta}}_k(\mathbf{x}_{t}) \rightarrow 0$. 

This recursion and dynamic progamming perspective also gives another significant outcome: one can optimize the policy $\pi$ by following a similar iteration rule. Recall the process in dynamic programming, the value of a specific state $\mathbf{x}_t$ can be updated by the best it can achieve based on $r$ and the current estimate of the return from $\mathbf{x}_{t+1}$. This essentially turns [](#eq-valuefunc) to an iteration rule by replacing the operator of $\mathbb{E}_{\pi_{\theta}(\mathbf{u}_t | \mathbf{x}_t )}$ to $\max\limits_{\mathbf{u}_t}$:

$$
\label{eq-valueitr}
\hat{V}^{\pi_{k+1}}(\mathbf{x}_t) \leftarrow \max\limits_{\mathbf{u}_t} [ r(\mathbf{x}_t, \mathbf{u}_t) + \mathbb{E}_{p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)}(\hat{V}^{\pi_{k}}(\mathbf{x}_{t+1}))]
$$

Note that the maximum operator means $\hat{V}^{\pi_{k+1}}(\mathbf{x}_t) \geq \hat{V}^{\pi_{k}}(\mathbf{x}_t)$, i.e. $\pi_{k+1}$ is at least as performant as $\pi_{k}$ or say it is guaranteed to get an improvement over the last iteration. This is called _value iteration_ and it provides methods different from policy-gradient for solving the decision-making problem. The issue of solely having the optimal $V^{\pi^*}$ is that it does not tell the agent how to act according to the optimal policy. Indeed, one may use it to design a policy with the recursion in [](#eq-valuefunc) as $\pi^*(\mathbf{x}_t) = \arg\max\limits_{\mathbf{u}_t} [ r(\mathbf{x}_t, \mathbf{u}_t) + \mathbb{E}_{p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)}(V^{\pi^*}(\mathbf{x}_{t+1}))]$. Evaluating this relies on knowing $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ which contradicts the general RL assumption. The fix to this is to use a _value-action_ function evaluated on both state and action. The function can be associated to $V^{\pi_{\theta}}$ as $Q^{\pi_{\theta}}(\mathbf{x}_t, \mathbf{u}_t) = r(\mathbf{x}_t, \mathbf{u}_t) + \mathbb{E}_{p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)}(V^{\pi_{\theta}}(\mathbf{x}_{t+1}))$. Having a look at the definition of value function, the $Q$-function $Q^{\pi_{\theta}}(\mathbf{x}, \mathbf{u})$ reads as _starting from state $\mathbf{x}$ and taking action $\mathbf{u}$, and then following the policy $\pi_{\theta}$ for remaining steps_. And for optimal policy $\pi^*$, it is easy to see that $V^{\pi^*}(\mathbf{x}) = \max\limits_{u}Q^{\pi^*}(\mathbf{x}, \mathbf{u})$. The iteration rule of [](#eq-valueitr) can then be similarly written to a form with $Q$-function:

$$
\label{eq-qlearning}
\hat{Q}^{\pi_{k+1}}(\mathbf{x}_t, \mathbf{u}_t) \leftarrow r(\mathbf{x}_t, \mathbf{u}_t) + \max\limits_{\mathbf{u}_{t+1}} \hat{Q}^{\pi_{k}}(\mathbf{x}_{t+1}, \mathbf{u}_{t+1})
$$

This algorithm is called _Q-learning_ and its variants represent a strand of value-based methods for solving RL problems. Using the optimal policy $\pi^*(\mathbf{x}_t) = \arg\max\limits_{\mathbf{u}_t} Q^{\pi^*}(\mathbf{x}_t, \mathbf{u}_t)$ removes the needs of knowing the transition model but introduces an optimization problem. For problems with a discrete action space, e.g. Go game, one can simply enumerate $\mathbf{u} \in \mathcal{U}$. For continous action space, a function (often called actor) approximating the maximum operation is trained at the same time for the ease of executing the policy. In the era of deep learning, neural networks are commonly used to parameterize $V^{\pi_{\theta}}$ and $Q^{\pi_{\theta}}$. This leads to popular algorithms based on value functions such as DQN[@dqn2016] and Soft Actor Critic (SAC)[@sac2018].

## Solving RL - Model-based Methods

The above model-free methods do not require to know the exact transition model $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ but relying on taking samples from it. This could have significant implications when the state transition involves real physical world like the learning to bike example:
* The samples can be extremely expensive to acquire and the rate of sample collection is upper-bounded by the real world clock.
* It could be risky to collect samples by physically interacting with the real world. This is especially problematic when we have yet learned a good $\pi_{\theta}$ for reasonable behaviours.
* Related to the point above, it could be very hard to autonomously "reset" to $\mathbf{x}_0 \sim \rho_0(\mathbf{x}_0)$ for new rollout collection, entailing a recovery policy that could handle any robot failing states in the real world.

All these observations point to using a proxy of real $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$, such as a simulator, to alleviate sampling costs and safety concerns. Besides using high-fidelity simulation, one might also think of fitting a surrogate of $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$, ${\color{red}\hat{p}}(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$, sometimes can be as complex as a _world model_ [@worldmodel2018], by regressing on the rollout data $\{ \mathbf{x}_{t+1}, \mathbf{x}_t, \mathbf{u}_t \}$. The RL objective thus becomes more amenable to familiar gradient-based optimization or other _model-based_ search that can exploit predictions from $\hat{p}(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$. Specifically, if the model of transition is deterministic, i.e. $\hat{p}(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t) \rightarrow \mathbf{x}_{t+1} = f(\mathbf{x}_t, \mathbf{u}_t)$, and the initial state is always $\mathbf{x}_0$, solving the RL objective can be formulated as:

$$
\label{eq-trajopt}
\begin{aligned}
&  \arg\max\limits_{\theta} \sum\limits_t r(\mathbf{x}_t,  \pi_{\theta}(\mathbf{x}_t)) \\
\text{s.t.} \quad & \mathbf{x}_{t+1} = f(\mathbf{x}_t, \pi_{\theta}(\mathbf{x}_t))  \quad \forall t = 0, 1, ..., T-1
\end{aligned}
$$
which is an optimization with the transition model defining a set of dynamic constraints for all time steps. When the policy is deterministic, the states are fully determined by $\mathbf{x}_0$ and $f(\mathbf{x}_t, \mathbf{u}_t)$. One can hence run a policy optimization without expensive queries to the real world. A further boost is from _automatic differentiation_ and _differentiable simulation_ with which the derivatives of $f(\mathbf{x}_t, \mathbf{u}_t)$ can be efficiently evaluated, making it very handy to plug the learned model and its derivatives to numerical optimization packages.

This may create an illusion that RL is no longer that hard after learning the transition model and particularly, a lot of machine learning techniques are known to be effective on regression. However, fitting $\hat{p}(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ to accurately replicate $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ may not be as trivial as it appears: 
* The model needs to generalize outside the rollouts it is trained upon.
* $\{ \mathbf{x}_{t+1}, \mathbf{x}_t, \mathbf{u}_t \}$ can be collected in a dependent way, e.g. by running a behaviour policy, so the occurance of input states will break the common independent-identical-distribution (i.i.d.) assumption of many regression methods.
* The model can perform catastrophically for long-horizon prediction even it is doing well on fitting pairs $\{ \mathbf{x}_{t+1}, \mathbf{x}_t, \mathbf{u}_t \}$ i.e. performing one-step prediction. Small errors from each step can be quickly accumulated (compounding errors) and may yield arbitrarily large deviation from the real sequence from $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$. 

As a result, many methods resort to regressing the outcome of recursively applying the model for $k$ steps, if data like $\{ \mathbf{x}_{t+1:t+k}, \mathbf{x}_t, \mathbf{u}_{t:t+k-1} \}$ are available, and merging the prediction from multiple models that are trained for different time scales.

In the end, even if the accuracy of the fitted model or simulation is fine, it is legit to ask whether a policy trained with $\hat{p}(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$ can work for $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$, because the ultimate target is acting in the real world. The answer is unfortunately a no for lots of interesting applications. As many other machine learning methods, a model (in this case the RL policy $\pi_{\theta}$) trained on one data distribution (in this case data from $\hat{p}(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$) would generally struggle on a different data distribution (in this case the real world data from $p(\mathbf{x}_{t+1}|\mathbf{x}_t, \mathbf{u}_t)$). Building more accurate simulation for better reality alignment is definitely a way to fight the issue. The other idea is from the policy perspective, e.g., learning policies that can work across domains. This touches machine learning research on transfer learning and domain adaptation, and more specifically _sim-to-real_ when the transfer concerns simulation and real world. We will delve more into this in [the latter module](../7_SimRL/notebooks.ipynb). 

```{figure} ../images/anymal_sim2real.png
---
height: 300px
name: anymal-sim2real
---
Using policy trained on simulated data to control real quadruped robot trepassing complex terrains. Sim2Real is realized by learning a policy that works for a range of simulation parameters for a better coverage of the real data distribution {cite:p}`anymal_scirob2018`.
```

## Software Practice for RL
<!-- Task/environment definition. Open AI Gym/Gymnasium API standards, link them to the RL diagram. Core at the environment, simulation as a surrogate to real-world. Repositories of task envs and algorithms.   -->
There are software packages for describing and solving RL problems. For the former, [Open AI Gym](https://github.com/openai/gym) and its successor [Gymnasium](https://gymnasium.farama.org) set the standard of programming the elements of $\{\mathcal{X}, \mathcal{U}, \mathcal{T}, \mathcal{R}, \rho_0 \}$ as reinforcement learning _environments_. Specifically, they define a few variables and functions so external processes, such as a policy function or a learning algorithm, can interact with the target problem.

In below is a simple code snippet defining a [CartPole balancing problem](https://en.wikipedia.org/wiki/Inverted_pendulum). It is clear to see the correspondance between RL elements and class member variable/functions. For instance, `reset()` is basically working as $\rho_0$ to sample an initial state for starting a brand new rollout.

In [1]:
import os
os.environ.pop('SIRL_USE_JAX', None)
from articulated_dynamics.math_utils import nplib
from articulated_dynamics.dynamics import Dynamics
from articulated_dynamics.robots import CartPole
from articulated_dynamics.integrator import integrate_euler

import numpy as np
import gymnasium as gym

class MyCartPoleEnv(gym.Env):
    def __init__(self) -> None:
        super().__init__()

        #define the state and action space
        #Box indicate a space defined by intervals with low/high bounds. Could be unbounded as well.
        self.observation_space=gym.spaces.Box(low=-np.inf, high=np.inf, shape=(4,))   
        #self.action_space=gym.spaces.Box(low=-10, high=10, shape=(1,))
        self.action_space=gym.spaces.Discrete(2)

        #prepare a CartPole dynamic model as the target to control
        #this should be an interface to a physical/simulation system
        self.model = CartPole()
        self.state = None
        self.t = 0
        self.horizon = 500

    def reset(self, seed=None):
        super().reset(seed=seed)

        #reset the environment: sampling a new initial state according to /pho; and reset the clock
        #there are 4 numbers characterising the state of a CartPole system: [CartTranslationalPosition, PoleJointPosition, CartTranslationalVelocity, PoleJointVelocity]
        #strictly speaking, the state should also include time index for MDP setup of finite horizon problems
        #here this is ignored so it is really a "partial observation" to the true state
        self.state = np.zeros(4)
        #only randomize the angular position of pole to a certain range
        self.state[1] = self.np_random.uniform(low=-np.pi/6, high=np.pi/6)

        self.t = 0
        self.dt = 0.01
        return self.state, {}

    def reward(self, s, a):
        #try to get to the up-right position
        pole_ang = s[1] % (2*np.pi)
        return 1 if pole_ang < np.deg2rad(10) or pole_ang > np.deg2rad(350) else 0
    
    def step(self, action):
        #evaluate the reward
        r = self.reward(self.state, action)

        #advance one step for the transition of environment
        #this will be interfacing control signal to the physical system and waiting the feedback after a certain amount of time
        #for simulation, one just do the calculation with the engine interface
        q = nplib.array(self.state[:2])
        qd = nplib.array(self.state[2:])
        #for cartpole, one only gets to apply control at the translational joint while the pole angular joint is passive
        #tau = nplib.array([action[0], 0])
        if action == 1:
            tau = nplib.array([10, 0])
        else:
            tau = nplib.array([-10, 0])

        qdd = Dynamics.forward(self.model, q, qd, tau)

        #integrate for a small time step, 0.01s here, for the next state
        q_next, qd_next = integrate_euler(self.model, self.dt, q, qd, qdd)

        #type conversion to numpy state format
        self.state = np.concatenate([np.array(q_next), np.array(qd_next)])

        #tick the time step and check if the rollout has reached the end
        self.t += 1
        terminated = ( self.t >= self.horizon ) or ( self.state[1] % (2*np.pi) >  np.deg2rad(20) and  self.state[1] % (2*np.pi) < np.deg2rad(340) ) 

        #depending the gym version, should return the finishing signal as solely a Done or more concrete terminated or truncated
        #see https://farama.org/Gymnasium-Terminated-Truncated-Step-API
        #the last dictionary allows passing some extra info when needed
        return self.state, r, terminated, False, {}

The lenghty `step()` function corresponding to state transition $\mathcal{T}$ is obviously the core description. This is usually an interface to convert the policy action and call an underlying simulation, e.g. for character animation, or wrapping the driver software of a system, e.g. a physical robot arm. 

Here, the CartPole environment is kinda a "Hello World" among RL problems. It is about moving a cart block horizontally such that a pole connecting to the cart through a revolute joint can keep upward. Look at the implementation of `reward()` to see how such a goal is expressed as a scalar. You may have many ideas of designing other rewards for the same purpose. The problem is solely about the simulation itself so no access to external motors/sensors are needed here (and make it runnable on a computer). Building simulation for such a system, e.g. rigid bodies linked through articulations, will make RL applicable to many interesting applications such as robotics and character animation. The course will cover some basics about articulated-body dynamics algorithms in [the module after](../3_ArticulatedDynamics/notebooks.ipynb).

With these interfaces, the RL environment can be simply stepped, e.g. to collect a state trajectory. One can animate the trajectory to see how the CartPole system moves:

In [6]:
#lets run the environment for a few steps and animate
env = MyCartPoleEnv()

#let the system evolve under random control
env.reset(seed=0)
state_traj = [env.state]
ret = 0
terminated = False

for i in range(env.horizon):
    state_next, r, terminated, truncated, info = env.step(env.action_space.sample())
    state_traj.append(state_next)
    ret += r

#visualize the rollout with an animation, only position part is needed
import articulated_dynamics.visualizer as viz
viewer = viz.P3JSViewer(width=400, height=300)
viewer.create_shapes(env.model)
pos_traj = nplib.array(state_traj)[:, :2]
viewer.place_shapes_qpos(env.model, pos_traj[0])

viewer.show()
anim = viewer.animate_qpos_traj(env.model, pos_traj, env.dt)
anim

Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(DirectionalLight(color='white', intensi…

AnimationAction(clip=AnimationClip(duration=5.01, tracks=(VectorKeyframeTrack(name='scene/cart.position', time…

Describing an RL problem as a gym environment allows to plug it into almost any RL algorithm libraries. The code block below shows using the implementation of PPO from [Stable Baselines3](https://github.com/DLR-RM/stable-baselines3) to solve the problem in less than one minute. The total step amount of interacting with the environment is set to 10000.

In [3]:
from stable_baselines3 import PPO

policy = PPO("MlpPolicy", env, verbose=0)
policy.learn(total_timesteps=10_000)

It is possible to log the learning progress to [Tensorboard](https://www.tensorflow.org/tensorboard) as other Deep Learning practice, e.g. by passing the learner a callback function. Through this, one may monitor how well the policy performs by recording the return of a few test rollouts or extra information that might be helpful for debugging. Refer to the document of corresponding RL libraries for details.

Similar to the environment animation code, the trained policy can be demonstrated to see whether the reward design and learning algorithms yield the desired behaviours:

In [5]:
obs, _ = env.reset(seed=0)
state_traj = []
ret = 0
for i in range(500):
    action, _ = policy.predict(obs, deterministic=True)
    state_next, r, terminated, truncated, info = env.step(action)
    obs = env.state
    state_traj.append(state_next)
    ret+=r

#visualize the rollout with an animation, only position part is needed
import articulated_dynamics.visualizer as viz
viewer = viz.P3JSViewer(width=400, height=300)
viewer.create_shapes(env.model)
pos_traj = nplib.array(state_traj)[:, :2]
viewer.place_shapes_qpos(env.model, pos_traj[0])

viewer.show()
anim = viewer.animate_qpos_traj(env.model, pos_traj, env.dt)
anim


Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(DirectionalLight(color='white', intensi…

AnimationAction(clip=AnimationClip(duration=5.0, tracks=(VectorKeyframeTrack(name='scene/cart.position', times…

:::{note}
Deep RL algorithms are usually with many parameters and approximation choices. The consequence to software implementation is that the same algorithm implemented in different libraries might have slight difference and not necessarily yield the exact same performance to the same gym environment. Actually, there is a study showing that implementation can matter quite significantly in applying Deep RL [@Engstrom2020Implementation]. 
:::


```{bibliography}
:filter: docname in docnames
```